# Pipeline Streaming sobre los 120 archivos JSONL (`usage_events_stream`)

Este notebook procesa los 120 archivos JSONL en 12 micro-lotes de 10 archivos (`maxFilesPerTrigger = 10`, 3.600 eventos por tanda), compara el resultado del watermark de 1 hora de la primera entrega contra los valores calibrados después del perfilado (`1368 hours` para p95 y `1440 hours` para la muestra completa) y comprueba que al volver a correr sobre el mismo `checkpointLocation` no se reprocesan los lotes ya confirmados.

In [1]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'config' / 'pipeline_config.json').exists() and (repo_root.parent / 'config' / 'pipeline_config.json').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.streaming_pipeline import run_streaming_pipeline

report = run_streaming_pipeline()
print(json.dumps(report['stream_source'], indent=2, ensure_ascii=False))

{
  "path": "data/raw/usage_events_stream/*.jsonl",
  "total_files": 120,
  "max_files_per_trigger": 10,
  "total_microbatch_triggers": 12,
  "events_per_trigger": 3600
}


## 1. Comparación de perfiles de watermark (`1 hour` vs. `1368 hours` vs. `1440 hours`)

In [2]:
print(json.dumps(report['watermark_profiles_comparison'], indent=2, ensure_ascii=False))

{
  "premature_1h_entrega1": {
    "delay_threshold": "1 hour",
    "accepted_valid_events": 3618,
    "quarantined_negative_cost_events": 216,
    "dropped_by_watermark_events": 39366,
    "valid_retention_pct": 8.42,
    "accepted_cost_usd": 13498.5863,
    "cost_lost_by_premature_watermark_usd": 134853.0146
  },
  "calibrated_p95_1368h_57d": {
    "delay_threshold": "1368 hours (57 days)",
    "accepted_valid_events": 41009,
    "quarantined_negative_cost_events": 216,
    "dropped_by_watermark_events": 1975,
    "valid_retention_pct": 95.41,
    "accepted_cost_usd": 141321.3079
  },
  "calibrated_replay_full_1440h_60d": {
    "delay_threshold": "1440 hours (60 days)",
    "accepted_valid_events": 42984,
    "quarantined_negative_cost_events": 216,
    "dropped_by_watermark_events": 0,
    "valid_retention_pct": 100.0,
    "accepted_cost_usd": 148351.6009,
    "accepted_carbon_kg": 365.5609,
    "accepted_genai_tokens": 2545481,
    "unit_imputed_count": 2069
  }
}


## 2. Verificación de checkpoint e idempotencia ante re-ejecución

In [3]:
print(json.dumps(report['checkpoint_fault_tolerance_and_idempotency'], indent=2, ensure_ascii=False))

{
  "checkpoint_location": "data/datalake/checkpoints/replay_full_1440h",
  "initial_run_executed_triggers": 12,
  "rerun_executed_triggers": 0,
  "rerun_skipped_committed_triggers": 12,
  "initial_accepted_events": 42984,
  "rerun_accepted_events": 42984,
  "initial_accepted_cost_usd": 148351.6009,
  "rerun_accepted_cost_usd": 148351.6009,
  "matches_batch_gold_cost_exactly": true,
  "is_checkpoint_idempotent": true
}


## 3. Detalle por micro-lote (primeros 4 lotes con `1440 hours` vs. `1 hour`)

In [4]:
print('--- Primeros 4 lotes con watermark de 1440 hours (60 días) ---')
print(json.dumps(report['replay_full_12_triggers_progression'][:4], indent=2, ensure_ascii=False))
print('\n--- Primeros 4 lotes con watermark de 1 hour (entrega 1) ---')
print(json.dumps(report['premature_1h_12_triggers_progression'][:4], indent=2, ensure_ascii=False))

--- Primeros 4 lotes con watermark de 1440 hours (60 días) ---
[
  {
    "batch_id": 0,
    "files_processed": "events_part_0000.jsonl..events_part_0009.jsonl",
    "files_count": 10,
    "input_rows": 3600,
    "batch_min_event_ts": "2025-07-03T00:08:00Z",
    "batch_max_event_ts": "2025-08-31T23:56:00Z",
    "watermark_before_trigger": null,
    "watermark_after_trigger": "2025-07-02T23:56:00Z",
    "accepted_rows": 3584,
    "quarantined_negative_cost_rows": 16,
    "dropped_by_watermark_rows": 0,
    "dropped_duplicates_rows": 0,
    "batch_accepted_cost_usd": 13380.906,
    "state_store_unique_ids_tracked": 3584
  },
  {
    "batch_id": 1,
    "files_processed": "events_part_0010.jsonl..events_part_0019.jsonl",
    "files_count": 10,
    "input_rows": 3600,
    "batch_min_event_ts": "2025-07-03T00:02:00Z",
    "batch_max_event_ts": "2025-08-31T23:51:00Z",
    "watermark_before_trigger": "2025-07-02T23:56:00Z",
    "watermark_after_trigger": "2025-07-02T23:56:00Z",
    "accepted_ro